# Silver conformed dimensions

Build 13 shared dimensions with explicit unknown members and publish the referential-integrity results.

In [ ]:
from pathlib import Path
import importlib
import sys

DATA_PRODUCT_PATH = Path("shared/integrated-test-data/projections/star-schema")
DATA_ROOT_CANDIDATES = [
    Path("/lakehouse/default/Files") / DATA_PRODUCT_PATH,
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / DATA_PRODUCT_PATH,
]

data_root = next(
    (
        candidate
        for candidate in DATA_ROOT_CANDIDATES
        if (candidate / "demo05_support.py").exists()
        and (candidate / "scenario_topology.csv").exists()
    ),
    None,
)
if data_root is None:
    raise FileNotFoundError("Shared star-schema projection was not found.")

if str(data_root) not in sys.path:
    sys.path.insert(0, str(data_root))
import demo05_support as demo

importlib.reload(demo)
sources = demo.load_sources(data_root)
dimensions = demo.build_dimensions(sources)
integrity_results = demo.validate_dimensions(sources, dimensions)
failed_checks = integrity_results.loc[~integrity_results["passed"]]
if not failed_checks.empty:
    raise ValueError(f"Dimension integrity validation failed:\n{failed_checks.to_string(index=False)}")

silver_tables = {**dimensions, "silver_integrity_results": integrity_results}
spark_session = globals().get("spark")
if spark_session is None:
    print("Spark is unavailable; validated Silver tables without publishing Delta tables.")
else:
    for table_name, frame in silver_tables.items():
        (
            spark_session.createDataFrame(demo.spark_compatible_frame(frame))
            .write.mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(table_name)
        )

print({table_name: len(frame) for table_name, frame in silver_tables.items()})
integrity_results